# 06. Shortcut learning & interpretability

Cần checkpoint `best.pt` của các config đã train (attach output của notebook 04/05 làm input nếu chạy Kaggle, rồi sửa `CKPT_DIR`).

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
import torch
from torch.utils.data import DataLoader
from PIL import Image
from vimmsd.data.dataset import ViMMSDCollator, build_datasets, load_all_records
from vimmsd.data.preprocessing import TextPreprocessor
from vimmsd.models.classifier import build_model
from vimmsd.utils.checkpoint import load_checkpoint
from vimmsd.analysis import interpretability as interp
from vimmsd.analysis.shortcut_check import run_shortcut_check

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = None  # None = <paths.output_dir>/<tên config>; Kaggle: thư mục output attach từ notebook trước


def load_trained(config):
    cfg = load_config(config)
    model = build_model(cfg)
    ckpt_dir = Path(CKPT_DIR) / cfg.name if CKPT_DIR else Path(cfg.paths.output_dir) / cfg.name
    load_checkpoint(ckpt_dir / "best.pt", model, map_location=device)
    model.to(device).eval()
    collator = ViMMSDCollator(model.tokenizer, model.image_processor, cfg.data.max_length,
                              cfg.data.get("max_caption_length"))
    return cfg, model, collator

## 1. Kiểm tra shortcut: tráo ảnh giữ nguyên text

Với các mẫu `image-sarcasm`/`multi-sarcasm`, nếu model thực sự dùng ảnh thì dự đoán phải đổi khi tráo ảnh. `flip_rate` thấp → model chủ yếu dựa vào text.

In [ ]:
import pandas as pd
rows = []
for config in ["configs/fusion_concat.yaml", "configs/fusion_cross_attn_focal.yaml"]:
    cfg, model, collator = load_trained(config)
    records = load_all_records(cfg)
    pre = TextPreprocessor.from_config(cfg.data.text)
    for pool_name, pool in [("tráo trong nhóm", None),
                            ("ảnh từ not-sarcasm", [r for r in records["train"] if r["label"] == 0])]:
        res = run_shortcut_check(model, records["test"], collator, list(cfg.data.labels), device,
                                 text_preprocessor=pre, image_pool=pool, cache_dir=cfg.paths.cache_dir)
        rows.append({"model": cfg.name, "cách tráo": pool_name, "n": res["n"], "flip_rate": res["flip_rate"],
                     "acc gốc": res["acc_original"], "acc sau tráo": res["acc_swapped"],
                     "prob shift": res["mean_prob_shift"]})
pd.DataFrame(rows).round(4)

## 2. Attention map (cross-attention) và Grad-CAM

In [ ]:
cfg, model, collator = load_trained("configs/fusion_cross_attn_focal.yaml")
labels = list(cfg.data.labels)
test_ds = build_datasets(cfg, splits=("test",))["test"]
idx = [i for i, r in enumerate(test_ds.records) if labels[r["label"]] in ("image-sarcasm", "multi-sarcasm")][:6]
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in collator([test_ds[i] for i in idx]).items()}

attn = interp.cross_attention_maps(model, batch)
cam = interp.gradcam(model, batch)
preds = attn["logits"].argmax(-1).tolist()

fig, axes = plt.subplots(len(idx), 3, figsize=(10, 3.4 * len(idx)))
for row, i, p, a_map, c_map in zip(axes, idx, preds, attn["image_heatmap"], cam["heatmap"]):
    rec = test_ds.records[i]
    img = Image.open(rec["image_path"]).convert("RGB")
    row[0].imshow(interp.encoder_view(img)); row[0].axis("off")
    row[0].set_title(f"thật: {labels[rec['label']]}\ndự đoán: {labels[p]}", fontsize=8)
    interp.overlay_heatmap(img, a_map, ax=row[1], title="cross-attention text→ảnh")
    interp.overlay_heatmap(img, c_map, ax=row[2], title="Grad-CAM")
plt.tight_layout(); plt.savefig("reports/figures/attention_maps.png", dpi=150); plt.show()

## 3. Từ được chú ý nhiều nhất trong caption

In [ ]:
for j, i in enumerate(idx):
    n = int(batch["attention_mask"][j].sum())
    words, scores = interp.merge_subwords(model.tokenizer, batch["input_ids"][j, :n].tolist(),
                                          attn["token_scores"][j, :n].tolist())
    print(f"[{labels[test_ds.records[i]['label']]}]")
    display(HTML(interp.words_to_html(words, scores)))